# Titanic Feature Engineering Notebook
## AI Assignment 2 - Full Pipeline

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier

In [ ]:
df = pd.read_csv('../data/train.csv')
df.head()

## 1. Data Cleaning

In [ ]:
df.isnull().sum()

In [ ]:
df['Age'].fillna(df['Age'].median(), inplace=True)
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
df['Cabin'] = df['Cabin'].fillna('Unknown')

## 2. Feature Engineering

In [ ]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

In [ ]:
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\\.', expand=False)
df['Deck'] = df['Cabin'].str[0]
df['Deck'] = df['Deck'].replace(np.nan, 'U')

In [ ]:
def age_group(age):
    if age < 13:
        return 'Child'
    elif age < 20:
        return 'Teen'
    elif age < 60:
        return 'Adult'
    else:
        return 'Senior'

df['AgeGroup'] = df['Age'].apply(age_group)

In [ ]:
df['FarePerPerson'] = df['Fare'] / df['FamilySize']

## 3. Encoding

In [ ]:
df = pd.get_dummies(df, columns=['Sex','Embarked','Title','Deck','AgeGroup'], drop_first=True)

## 4. Feature Selection

In [ ]:
X = df.drop(['Survived','Name','Ticket','Cabin'], axis=1)
y = df['Survived']

model = RandomForestClassifier()
model.fit(X,y)

feat_imp = pd.Series(model.feature_importances_, index=X.columns)
feat_imp.sort_values().plot(kind='barh')
plt.show()